# Metal complex builder for aryl electrophile activation (molSimplify + RDKit)

Builds starting structures for **any metal, oxidation state, and ligand set** reacting with aryl electrophiles
(Ar–Cl, Ar–Br, Ar–I, Ar–OTf, Ar–OTs, ...), for several mechanistic channels, and writes Gaussian + g-xTB
(`xtb-gaussian`) optimization inputs plus a SLURM array script.

For a starting complex **M(n)(spectator ligands)** and **Ar–X**, the channels are:

| Channel | Metal species built | Partner species built |
|---|---|---|
| `start` | M(n)(L) | Ar–X |
| `OA` | M(n+2)(L)(Ar)(X) — two-electron oxidative addition (concerted) | — |
| `OA_cation` | [M(n+2)(L)(Ar)]⁺ — ionic / SNAr-like OA, X⁻ separated | X⁻ |
| `XAT` | M(n+1)(L)(X) — halogen/group abstraction, inner-sphere ET | Ar• |
| `OSET` | [M(n+1)(L)]⁺ — outer-sphere electron transfer (same geometry as `start`) | [Ar–X]•⁻ |

Every structure is a **starting geometry only** (molSimplify uses templates and tabulated bond lengths).
Optimize with g-xTB, confirm zero imaginary frequencies, and look at each one before scans or TS searches.

**Setup (once):**
```bash
mamba create -n msimp -c conda-forge python=3.10 rdkit openbabel ipykernel pandas
conda activate msimp
pip install molSimplify py3Dmol
python -m ipykernel install --user --name msimp
```
molSimplify is not on conda-forge under that name; install it with pip as above. On native Windows, molSimplify
2.0.0 has a temp-file bug (`PermissionError: [WinError 32]`); WSL avoids it.

## 1. Settings

In [24]:
import os, sys
WORKDIR = os.path.abspath("complex_structures")
os.makedirs(WORKDIR, exist_ok=True)
MS_CMD = (sys.executable, "-m", "molSimplify")
SKIP_ANN = True            # tabulated bond lengths instead of molSimplify's neural network

# Isomers: "all" = try every distinct ordering of the monodentate ligands and keep the distinct isomers;
#          "first" = only vary which monodentate goes first (faster, fewer isomers)
ISOMERS = "all"
MAX_ORDERINGS = 12         # cap on molSimplify builds per species

# Gaussian + g-xTB settings: keep IDENTICAL for everything you compare
NPROC = 4
MEM = "16000MB"
XTB_METHOD = "--gxtb"      # or "--gfn2"; do not combine methods
XTB_ACC = "0.0001"
XTB_EXTRA = ""             # e.g. "--alpb dmf" if your build supports it with this method

## 2. Helper functions (run, no edits needed)

In [25]:
import os, re, glob, shutil, subprocess, itertools
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import GetPeriodicTable

PT = GetPeriodicTable()
HALOGENS = {"F", "Cl", "Br", "I"}
HALIDE_LIB = {"Cl": "chloride", "Br": "bromide", "I": "iodide"}
# group number for d-electron counting (d count = group - oxidation state)
GROUP = {"Sc":3,"Ti":4,"V":5,"Cr":6,"Mn":7,"Fe":8,"Co":9,"Ni":10,"Cu":11,"Zn":12,
         "Y":3,"Zr":4,"Nb":5,"Mo":6,"Tc":7,"Ru":8,"Rh":9,"Pd":10,"Ag":11,"Cd":12,
         "Hf":4,"Ta":5,"W":6,"Re":7,"Os":8,"Ir":9,"Pt":10,"Au":11,"Hg":12}
DEFAULT_GEOMETRY = {1: "no", 2: "li", 3: "tpl", 4: "sqp", 5: "spy", 6: "oct", 7: "pbp"}

# ---------------- ligand library ----------------

def load_ligand_library():
    """Read molSimplify's ligands.dict -> {name: {"denticity": n, "charge": q}}"""
    import molSimplify
    path = os.path.join(os.path.dirname(molSimplify.__file__), "Ligands", "ligands.dict")
    lib = {}
    for line in open(path):
        if ":" not in line or line.lstrip().startswith("#"):
            continue
        name, rest = line.split(":", 1)
        f = [s.strip() for s in rest.split(",")]
        try:
            lib[name.strip()] = {"denticity": len(f[2].split()), "charge": int(f[-1])}
        except (IndexError, ValueError):
            pass
    return lib

def resolve_ligand(key, d, library):
    """Fill in denticity and charge for a ligand definition (library name or SMILES + donors)."""
    d = dict(d)
    if "library" in d:
        if d["library"] not in library:
            raise KeyError(f"{key}: '{d['library']}' is not in molSimplify's ligand library")
        d.setdefault("denticity", library[d["library"]]["denticity"])
        d.setdefault("charge", library[d["library"]]["charge"])
    else:
        mol = Chem.MolFromSmiles(d["smiles"])
        if mol is None:
            raise ValueError(f"{key}: RDKit cannot parse SMILES {d['smiles']}")
        d.setdefault("charge", Chem.GetFormalCharge(mol))
        if "donors" not in d:
            raise ValueError(f"{key}: SMILES ligands need 'donors' (1-based heavy-atom indices)")
        d.setdefault("denticity", len(d["donors"]))
    return d

def smiles_atom_indices(smiles):
    """1-based heavy-atom indices in SMILES order (what molSimplify's -smicat uses)."""
    mol = Chem.MolFromSmiles(smiles)
    return [(a.GetIdx() + 1, a.GetSymbol(), a.GetFormalCharge(), a.GetIsAromatic()) for a in mol.GetAtoms()]

def guess_donor_atoms(smiles, elements=("N", "P")):
    """Suggest donor atoms (1-based): N/P atoms without H. ALWAYS verify by eye."""
    mol = Chem.MolFromSmiles(smiles)
    return [a.GetIdx() + 1 for a in mol.GetAtoms() if a.GetSymbol() in elements and a.GetTotalNumHs() == 0]

# ---------------- aryl electrophiles ----------------

KNOWN_LG = {"O=S(=O)([O-])C(F)(F)F": "OTf", "Cc1ccc(S(=O)(=O)[O-])cc1": "OTs",
            "CS(=O)(=O)[O-]": "OMs", "O=S(=O)([O-])C(F)(F)C(F)(F)C(F)(F)C(F)(F)F": "ONf"}

def split_aryl_electrophile(arx_smiles):
    """Ar-X (X = Cl/Br/I or a sulfonate) -> dict with aryl anion SMILES (ipso C = atom 1),
    aryl radical SMILES, leaving-group name, its ligand definition, its anion SMILES, and donor element."""
    mol = Chem.MolFromSmiles(arx_smiles)
    if mol is None:
        raise ValueError(f"RDKit cannot parse SMILES: {arx_smiles}")
    cuts = []
    for a in mol.GetAtoms():
        if not (a.GetSymbol() == "C" and a.GetIsAromatic()):
            continue
        for nb in a.GetNeighbors():
            if nb.GetSymbol() in HALIDE_LIB:
                cuts.append((a.GetIdx(), nb.GetIdx()))
            elif nb.GetSymbol() == "O" and nb.GetDegree() == 2:
                S = [s for s in nb.GetNeighbors() if s.GetSymbol() == "S"]
                if S and sum(1 for o in S[0].GetNeighbors() if o.GetSymbol() == "O") >= 3:
                    cuts.append((a.GetIdx(), nb.GetIdx()))
    if len(cuts) != 1:
        raise ValueError(f"{arx_smiles}: expected exactly one aryl C-X bond (Cl/Br/I/sulfonate), found {len(cuts)}")
    ci, xi = cuts[0]
    def cut(charge_c, radical_c):
        rw = Chem.RWMol(mol); rw.RemoveBond(ci, xi)
        for idx, tag, chg, rad in ((ci, 1, charge_c, radical_c), (xi, 2, -1, 0)):
            at = rw.GetAtomWithIdx(idx)
            at.SetFormalCharge(chg); at.SetNumRadicalElectrons(rad)
            at.SetNoImplicit(True); at.SetNumExplicitHs(0); at.SetIntProp("cut", tag)
        m = rw.GetMol(); Chem.SanitizeMol(m)
        out = {}
        for f in Chem.GetMolFrags(m, asMols=True, sanitizeFrags=True):
            for at in f.GetAtoms():
                if at.HasProp("cut"):
                    out[at.GetIntProp("cut")] = Chem.MolToSmiles(f, rootedAtAtom=at.GetIdx())
        return out
    anion = cut(-1, 0); radical = cut(0, 1)
    aryl, lg = anion[1], anion[2]
    assert Chem.MolFromSmiles(aryl).GetAtomWithIdx(0).GetFormalCharge() == -1, aryl
    sym = mol.GetAtomWithIdx(xi).GetSymbol()
    if sym in HALIDE_LIB:
        name, ldef, el = sym, {"library": HALIDE_LIB[sym]}, sym
    else:
        name = KNOWN_LG.get(Chem.MolToSmiles(Chem.MolFromSmiles(lg)), "OSO2R")
        ldef, el = {"smiles": lg, "donors": [1]}, "O"
    return {"aryl": aryl, "aryl_radical": radical[1], "X": name, "Xdef": ldef, "X_anion": lg, "X_element": el}

# ---------------- organic / small species ----------------

def organic_xyz(smiles, path, seed=42):
    """3D structure for a metal-free species (Ar-X, Ar radical, X- anion) via RDKit."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    if mol.GetNumAtoms() == 1:
        a = mol.GetAtomWithIdx(0)
        with open(path, "w") as f:
            f.write(f"1\n{smiles}\n{a.GetSymbol()} 0.000000 0.000000 0.000000\n")
        return path
    params = AllChem.ETKDGv3(); params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError(f"Embedding failed for {smiles}")
    try:
        if AllChem.MMFFHasAllMoleculeParams(mol): AllChem.MMFFOptimizeMolecule(mol)
        else: AllChem.UFFOptimizeMolecule(mol)
    except Exception:
        pass   # radicals etc.: keep the ETKDG geometry, xTB will optimize it
    with open(path, "w") as f:
        f.write(Chem.MolToXYZBlock(mol))
    return path

# ---------------- electrons, spin ----------------

def total_electrons(el, charge):
    return sum(PT.GetAtomicNumber(e) for e in el) - charge

def allowed_mults(el, charge, mults):
    """Keep only multiplicities consistent with the electron count (even electrons -> odd multiplicity)."""
    ne = total_electrons(el, charge)
    return [m for m in mults if (ne % 2 == 0) == (m % 2 == 1)], ne

def d_count(metal, ox):
    return GROUP[metal] - ox if metal in GROUP else None

def spin_options(metal, ox):
    """All possible multiplicities from low to high spin for a d^n metal (single-metal, ligand-closed-shell)."""
    n = d_count(metal, ox)
    if n is None or n < 0: return [1, 2]
    unpaired_max = n if n <= 5 else 10 - n
    return sorted({u + 1 for u in range(unpaired_max % 2, unpaired_max + 1, 2)})

# ---------------- molSimplify ----------------

def write_ms_input(job, ligand_db, rundir, skipANN=True):
    libnames, occ, smicat = [], [], []
    for key, count in job["ligands"]:
        L = ligand_db[key]
        if "library" in L:
            libnames.append(L["library"])
        else:
            libnames.append(L["smiles"])
            smicat.append("[" + ",".join(str(i) for i in L["donors"]) + "]")
        occ.append(str(count))
    lines = [f"-rundir {rundir}", f"-core {job['metal']}", f"-geometry {job['geometry']}",
             f"-coord {job['coord']}", f"-oxstate {job['oxstate']}", f"-spin {job['ms_mult']}",
             "-lig " + " ".join(libnames), "-ligocc " + " ".join(occ), f"-name {job['name']}"]
    if smicat: lines.append("-smicat " + ",".join(smicat))
    if skipANN: lines.append("-skipANN True")
    path = os.path.join(rundir, f"{job['name']}.in")
    with open(path, "w") as f: f.write("\n".join(lines) + "\n")
    return path

def run_molsimplify(job, ligand_db, workdir, ms_cmd=("molsimplify",), skipANN=True, timeout=600):
    rundir = os.path.abspath(os.path.join(workdir, "ms_runs", job["name"]))
    if os.path.isdir(rundir): shutil.rmtree(rundir)   # molSimplify prompts if the folder exists
    os.makedirs(rundir)
    inp = write_ms_input(job, ligand_db, rundir, skipANN=skipANN)
    res = subprocess.run(list(ms_cmd) + ["-i", inp], cwd=rundir, stdin=subprocess.DEVNULL,
                         capture_output=True, text=True, timeout=timeout)
    with open(os.path.join(rundir, "molsimplify.log"), "w") as f:
        f.write(res.stdout + "\n--- stderr ---\n" + res.stderr)
    xyzs = sorted(glob.glob(os.path.join(rundir, "**", "*.xyz"), recursive=True), key=os.path.getmtime)
    if not xyzs:
        raise RuntimeError(f"{job['name']}: no xyz produced; see {rundir}/molsimplify.log")
    return xyzs[-1]

# ---------------- geometry checks ----------------

def read_xyz(path):
    L = open(path).read().splitlines(); n = int(L[0].split()[0])
    rows = [l.split() for l in L[2:2 + n]]
    return [r[0] for r in rows], np.array([[float(v) for v in r[1:4]] for r in rows])

def write_xyz(path, el, x, comment=""):
    with open(path, "w") as f:
        f.write(f"{len(el)}\n{comment}\n")
        for e, (a, b, c) in zip(el, x): f.write(f"{e:<3s}{a:14.6f}{b:14.6f}{c:14.6f}\n")

def donor_cutoff(metal, e):
    """Bonding cutoff: 1.3 x (covalent radius sum), generous enough for long M-I / M-P bonds."""
    return 1.3 * (PT.GetRcovalent(metal) + PT.GetRcovalent(e))

def metal_environment(el, x, metal):
    m = el.index(metal)
    d = np.linalg.norm(x - x[m], axis=1)
    nb = [i for i in range(len(el)) if i != m and el[i] != "H" and d[i] < donor_cutoff(metal, el[i])]
    D = np.linalg.norm(x[:, None] - x[None], axis=-1); np.fill_diagonal(D, 9)
    return m, nb, d, int((D < 0.9).sum() // 2)

def angle(x, c, i, j):
    v1, v2 = x[i] - x[c], x[j] - x[c]
    return float(np.degrees(np.arccos(np.clip(v1 @ v2 / np.linalg.norm(v1) / np.linalg.norm(v2), -1, 1))))

def trans_pairs(el, x, m, nb, labels=None, cutoff=140.0):
    """Pairs of metal-bound atoms roughly trans to each other (> cutoff deg), as label strings."""
    lab = labels or {i: el[i] for i in nb}
    used, pairs = set(), []
    cand = sorted(((angle(x, m, i, j), i, j) for i, j in itertools.combinations(nb, 2)), reverse=True)
    for a, i, j in cand:
        if a < cutoff: break
        if i in used or j in used: continue
        used |= {i, j}; pairs.append("-".join(sorted([lab[i], lab[j]])))
    return sorted(pairs)

def find_ipso(el, x, m, nb):
    """Metal-bound C whose heavy-atom neighbours are exactly two C (aryl ipso), else nearest bound C."""
    Cs = [i for i in nb if el[i] == "C"]
    def cneigh(i):
        return [j for j in range(len(el)) if j != i and el[j] != "H" and j != m
                and np.linalg.norm(x[i] - x[j]) < 1.6]
    aryl = [i for i in Cs if len(cneigh(i)) == 2 and all(el[j] == "C" for j in cneigh(i))]
    pool = aryl or Cs
    return min(pool, key=lambda i: np.linalg.norm(x[i] - x[m])) if pool else None

def find_leaving_atom(el, x, m, nb, X_element, ipso):
    """Metal-bound atom of the leaving group, preferring one cis to the ipso C."""
    if X_element == "O":   # sulfonate O: bound O with an S neighbour
        cands = [i for i in nb if el[i] == "O" and any(el[j] == "S" and np.linalg.norm(x[i] - x[j]) < 1.8
                                                      for j in range(len(el)))]
    else:
        cands = [i for i in nb if el[i] == X_element]
    if not cands: return None
    if ipso is None: return cands[0]
    return min(cands, key=lambda i: angle(x, m, ipso, i))   # most cis to the aryl

def reorder_key_atoms(el, x, metal, X_element=None, with_aryl=True):
    """Metal = 1, ipso C = 2 (if aryl), leaving-group atom = next, then the rest. Returns el, x, key dict."""
    m, nb, d, _ = metal_environment(el, x, metal)
    order, key = [m], {metal: 1}
    ipso = find_ipso(el, x, m, nb) if with_aryl else None
    if ipso is not None:
        order.append(ipso); key["C_ipso"] = len(order)
    if X_element:
        xa = find_leaving_atom(el, x, m, nb, X_element, ipso)
        if xa is not None:
            order.append(xa); key["X_leaving"] = len(order)
    order += [i for i in range(len(el)) if i not in order]
    return [el[i] for i in order], x[order], key

# ---------------- Gaussian + xTB input ----------------

def gaussian_xtb_input(name, el, x, charge, mult, nproc=4, mem="16000MB", acc="0.0001",
                       method_flags="--gxtb", extra_xtb=""):
    ext = f"xtb-gaussian -P {nproc} {method_flags} --charge {charge} --uhf {mult - 1} --acc {acc} {extra_xtb}".strip()
    job = "opt=(maxcycles=200,nomicro) freq=noraman" if len(el) > 1 else "sp"
    coords = "\n".join(f"{e:<3s}{a:14.6f}{b:14.6f}{c:14.6f}" for e, (a, b, c) in zip(el, x))
    return (f"%mem={mem}\n%chk={name}.chk\n# external=\"{ext}\" UGBS {job}\n\n"
            f"{name}\n\n{charge} {mult}\n{coords}\n\n")

## 3. molSimplify's ligand library
Charges and denticities of library ligands are read from it automatically.

In [26]:
LIB = load_ligand_library()
print(f"{len(LIB)} library ligands")
for n in ["terpy", "bipy", "phen", "pph3", "pcy3", "pme3", "acac", "bromide", "chloride", "iodide", "pyridine", "acetonitrile"]:
    print(f"  {n:13s}", LIB.get(n, "NOT FOUND"))
# search: sorted(n for n in LIB if "py" in n)

255 library ligands
  terpy         {'denticity': 3, 'charge': 0}
  bipy          {'denticity': 2, 'charge': 0}
  phen          {'denticity': 2, 'charge': 0}
  pph3          {'denticity': 1, 'charge': 0}
  pcy3          {'denticity': 1, 'charge': 0}
  pme3          {'denticity': 1, 'charge': 0}
  acac          {'denticity': 2, 'charge': -1}
  bromide       {'denticity': 1, 'charge': -1}
  chloride      {'denticity': 1, 'charge': -1}
  iodide        {'denticity': 1, 'charge': -1}
  pyridine      {'denticity': 1, 'charge': 0}
  acetonitrile  {'denticity': 1, 'charge': 0}


## 4. Your inputs

**`LIGANDS`**: every spectator ligand, by library name or SMILES. For SMILES give `donors` = 1-based heavy-atom
positions in the SMILES that bind the metal (check them with `smiles_atom_indices`). Charge and denticity are
filled in automatically.

**`SYSTEMS`**: each starting complex: metal, oxidation state, and spectator ligands with counts. Optional per-species
overrides: `geometry` (e.g. `{"start": "thd"}` for tetrahedral Cu(I)) and `mults` (e.g. `{"OA": [2]}`); otherwise
the geometry follows the coordination number (2 li, 3 tpl, 4 sqp, 5 spy, 6 oct) and all low-to-high-spin
multiplicities for that d-electron count are written.

**Tested:** Ni with terpy (tridentate) and Pd with PPh₃ (monodentate). Bidentate ligands with a 3-coordinate metal
built badly in earlier tests; check those rows carefully.

In [27]:
LIGANDS = {
    "terpy": {"library": "terpy"},
    # "terpy": {"smiles": "c1ccc(nc1)-c1cccc(n1)-c1ccccn1", "donors": [5, 12, 18]},   # same ligand as SMILES
    "Br":    {"library": "bromide"},
    # "Cl":    {"library": "chloride"},
    # "PPh3":  {"library": "pph3"},
}

SYSTEMS = [
    dict(name="terpyNiBr", metal="Ni", ox=1, spectators=[("terpy", 1), ("Br", 1)]),
    # dict(name="PdPPh3", metal="Pd", ox=0, spectators=[("PPh3", 2)]),
    # dict(name="CuPhen", metal="Cu", ox=1, spectators=[("phen", 1), ("Br", 1)], geometry={"start": "tpl"}),
]

ARYL_ELECTROPHILES = {
    "PhI": "Ic1ccccc1",
    # "PhBr":  "Brc1ccccc1",
    # "PhCl":  "Clc1ccccc1",
    # "PhOTf": "FC(F)(F)S(=O)(=O)Oc1ccccc1",
    # "PhOTs": "Cc1ccc(cc1)S(=O)(=O)Oc1ccccc1"
}

CHANNELS = ["OA"]
# CHANNELS = ["OA", "OA_cation", "XAT", "OSET"]     # "start" is always built

In [28]:
# Resolve ligands (charge, denticity) and show donor atoms of SMILES ligands
for key in list(LIGANDS):
    LIGANDS[key] = resolve_ligand(key, LIGANDS[key], LIB)
    L = LIGANDS[key]
    src = L.get("library") or L["smiles"]
    print(f"{key:8s} denticity {L['denticity']}  charge {L['charge']:+d}  ({src})")
    if "smiles" in L:
        for idx, sym, chg, arom in smiles_atom_indices(L["smiles"]):
            if idx in L["donors"]: print(f"            donor atom {idx}: {sym}")

terpy    denticity 3  charge +0  (terpy)
Br       denticity 1  charge -1  (bromide)


## 5. Build the job list

In [29]:
SPLIT = {name: split_aryl_electrophile(s) for name, s in ARYL_ELECTROPHILES.items()}
ligand_db = {k: v for k, v in LIGANDS.items()}
for name, s in SPLIT.items():
    ligand_db[f"Ar_{name}"] = {"smiles": s["aryl"], "donors": [1], "denticity": 1, "charge": -1}
    ligand_db[s["X"]] = resolve_ligand(s["X"], s["Xdef"], LIB)
    print(f"{name}: aryl {s['aryl']} + leaving group {s['X']} (binds through {s['X_element']})")

def orderings(multi, mono):
    """Ligand sequences: multidentates first (fixed), then orderings of the monodentates."""
    mono_perms = sorted(set(itertools.permutations(mono)))
    if ISOMERS == "first":
        seen, keep = set(), []
        for p in mono_perms:
            if p[0] not in seen: seen.add(p[0]); keep.append(p)
        mono_perms = keep
    out = []
    for p in mono_perms[:MAX_ORDERINGS]:
        seq = [(k, c) for k, c in multi]
        for k in p:
            if seq and seq[-1][0] == k and ligand_db[k]["denticity"] == 1: seq[-1] = (k, seq[-1][1] + 1)
            else: seq.append((k, 1))
        out.append(seq)
    return out

def species(sysd, kind, ox, extra, arname=None):
    """One species: spectators + extra ligands (list of keys) at oxidation state ox."""
    multi = [(k, c) for k, c in sysd["spectators"] if ligand_db[k]["denticity"] > 1]
    mono = [k for k, c in sysd["spectators"] if ligand_db[k]["denticity"] == 1 for _ in range(c)] + extra
    cn = sum(ligand_db[k]["denticity"] * c for k, c in multi) + len(mono)
    charge = ox + sum(ligand_db[k]["charge"] * c for k, c in multi) + sum(ligand_db[k]["charge"] for k in mono)
    geom = sysd.get("geometry", {}).get(kind) or DEFAULT_GEOMETRY[cn]
    mults = sysd.get("mults", {}).get(kind) or spin_options(sysd["metal"], ox)
    base = f"{sysd['name']}_{kind}" + (f"_{arname}" if arname else "")
    return [dict(name=f"{base}_o{i+1}", group=base, system=sysd["name"], kind=kind, arname=arname,
                 metal=sysd["metal"], oxstate=ox, charge=charge, mults=mults, ms_mult=mults[0],
                 geometry=geom, coord=cn, ligands=seq)
            for i, seq in enumerate(orderings(multi, mono))]

jobs = []
for S in SYSTEMS:
    n = S["ox"]
    jobs += species(S, "start", n, [])
    for arname, s in SPLIT.items():
        if "OA" in CHANNELS:        jobs += species(S, "OA", n + 2, [f"Ar_{arname}", s["X"]], arname)
        if "OA_cation" in CHANNELS: jobs += species(S, "OA_cation", n + 2, [f"Ar_{arname}"], arname)
        if "XAT" in CHANNELS:       jobs += species(S, "XAT", n + 1, [s["X"]], arname)
print(f"{len(jobs)} molSimplify builds:")
for j in jobs:
    print(f"  {j['name']:32s} {j['metal']}({j['oxstate']}) CN {j['coord']} {j['geometry']:4s} charge {j['charge']:+d} "
          f"mults {j['mults']}  {j['ligands']}")

PhI: aryl [c-]1ccccc1 + leaving group I (binds through I)
7 molSimplify builds:
  terpyNiBr_start_o1               Ni(1) CN 4 sqp  charge +0 mults [2]  [('terpy', 1), ('Br', 1)]
  terpyNiBr_OA_PhI_o1              Ni(3) CN 6 oct  charge +0 mults [2, 4]  [('terpy', 1), ('Ar_PhI', 1), ('Br', 1), ('I', 1)]
  terpyNiBr_OA_PhI_o2              Ni(3) CN 6 oct  charge +0 mults [2, 4]  [('terpy', 1), ('Ar_PhI', 1), ('I', 1), ('Br', 1)]
  terpyNiBr_OA_PhI_o3              Ni(3) CN 6 oct  charge +0 mults [2, 4]  [('terpy', 1), ('Br', 1), ('Ar_PhI', 1), ('I', 1)]
  terpyNiBr_OA_PhI_o4              Ni(3) CN 6 oct  charge +0 mults [2, 4]  [('terpy', 1), ('Br', 1), ('I', 1), ('Ar_PhI', 1)]
  terpyNiBr_OA_PhI_o5              Ni(3) CN 6 oct  charge +0 mults [2, 4]  [('terpy', 1), ('I', 1), ('Ar_PhI', 1), ('Br', 1)]
  terpyNiBr_OA_PhI_o6              Ni(3) CN 6 oct  charge +0 mults [2, 4]  [('terpy', 1), ('I', 1), ('Br', 1), ('Ar_PhI', 1)]


## 6. Run molSimplify, check structures, keep distinct isomers
Isomers are compared by which metal-bound atoms are trans to each other; repeats are marked `duplicate`.
Depending on how many structures you are generating, it takes a while (several minutes for ~10 structures)

In [30]:
import pandas as pd
results, signatures = [], {}
for j in jobs:
    row = {"name": j["name"], "group": j["group"]}
    try:
        p = run_molsimplify(j, ligand_db, WORKDIR, ms_cmd=MS_CMD, skipANN=SKIP_ANN)
        el, x = read_xyz(p)
        m, nb, d, clashes = metal_environment(el, x, j["metal"])
        sig = ", ".join(trans_pairs(el, x, m, nb)) or "-"
        row.update(status="ok", xyz=p, atoms=len(el), CN=len(nb), expected_CN=j["coord"],
                   M_L=", ".join(f"{el[i]} {d[i]:.2f}" for i in sorted(nb, key=lambda i: el[i])),
                   trans=sig, clashes=clashes, shortest_ML=round(min(d[i] for i in nb), 2) if nb else None)
        if len(nb) != j["coord"] or clashes or (nb and min(d[i] for i in nb) < 1.75):
            row["status"] = "CHECK"
        elif sig in signatures.setdefault(j["group"], set()):
            row["status"] = "duplicate"
        signatures[j["group"]].add(sig)
    except Exception as e:
        row.update(status="FAILED", error=str(e)[:120])
    results.append(row)
df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 70)
# df.drop(columns=["xyz", "group"], errors="ignore")
df[df.status != "duplicate"].drop(columns=["xyz", "group"], errors="ignore")

,name,status,atoms,CN,expected_CN,M_L,trans,clashes,shortest_ML
0,terpyNiBr_start_o1,ok,31,4,4,"Br 2.35, N 2.23, N 2.14, N 2.23","Br-N, N-N",0,2.14
1,terpyNiBr_OA_PhI_o1,ok,43,6,6,"Br 2.35, C 2.14, I 2.61, N 2.23, N 2.14, N 2.23","Br-I, C-N, N-N",0,2.14
3,terpyNiBr_OA_PhI_o3,ok,43,6,6,"Br 2.35, C 2.14, I 2.61, N 2.23, N 2.14, N 2.23","Br-N, C-I, N-N",0,2.14
5,terpyNiBr_OA_PhI_o5,ok,43,6,6,"Br 2.35, C 2.14, I 2.61, N 2.23, N 2.14, N 2.23","Br-C, I-N, N-N",0,2.14


`CN` should equal `expected_CN`, `clashes` 0, `shortest_ML` above ~1.8 Å. For CHECK/FAILED rows, look at the structure (section 7b) or `ms_runs/<name>/molsimplify.log`.

## 7. Write renumbered XYZ files, Gaussian/g-xTB inputs, and the SLURM script

In [31]:
OUT = os.path.join(WORKDIR, "gxtb_opt")
os.makedirs(OUT, exist_ok=True)
coms, index_lines = [], []

def emit(name, el, x, charge, mult, note=""):
    with open(os.path.join(OUT, name + ".com"), "w") as f:
        f.write(gaussian_xtb_input(name, el, x, charge, mult, NPROC, MEM, XTB_ACC, XTB_METHOD, XTB_EXTRA))
    write_xyz(os.path.join(OUT, name + ".xyz"), el, x, comment=f"{name} charge {charge} mult {mult} {note}")
    coms.append(name + ".com"); index_lines.append(f"{name}\tcharge {charge:+d} mult {mult}\t{note}")

def emit_all_mults(base, el, x, charge, mults, note=""):
    ok, ne = allowed_mults(el, charge, mults)
    bad = sorted(set(mults) - set(ok))
    if bad: print(f"  {base}: skipped multiplicities {bad} (impossible with {ne} electrons)")
    for m in ok: emit(f"{base}_m{m}", el, x, charge, m, note)

jobmap = {j["name"]: j for j in jobs}
for row in results:
    if row["status"] not in ("ok", "CHECK"): continue
    j = jobmap[row["name"]]
    el, x = read_xyz(row["xyz"])
    s = SPLIT.get(j["arname"]) if j["arname"] else None
    el, x, key = reorder_key_atoms(el, x, j["metal"], X_element=(s["X_element"] if s and j["kind"] in ("OA", "XAT") else None),
                                   with_aryl=j["kind"] in ("OA", "OA_cation"))
    note = f"key {key} trans [{row['trans']}]" + (" CHECK" if row["status"] == "CHECK" else "")
    if "C_ipso" in key and "X_leaving" in key and angle(x, 0, key["C_ipso"] - 1, key["X_leaving"] - 1) > 140:
        note += "  (X trans to Ar: not a direct concerted-OA product)"
    emit_all_mults(j["name"], el, x, j["charge"], j["mults"], note)
    if j["kind"] == "start" and "OSET" in CHANNELS:      # oxidized start complex, same geometry
        emit_all_mults(j["name"].replace("_start_", "_OSET_"), el, x, j["charge"] + 1,
                       spin_options(j["metal"], j["oxstate"] + 1), f"key {key} (start geometry, oxidized)")

# metal-free partners
for arname, s in SPLIT.items():
    smi = ARYL_ELECTROPHILES[arname]
    p = organic_xyz(smi, os.path.join(OUT, f"tmp_{arname}.xyz")); el, x = read_xyz(p); os.remove(p)
    emit(f"ArX_{arname}", el, x, 0, 1)
    if "OSET" in CHANNELS: emit(f"ArX_radanion_{arname}", el, x, -1, 2, "ArX geometry")
    if "XAT" in CHANNELS:
        p = organic_xyz(s["aryl_radical"], os.path.join(OUT, f"tmp_{arname}.xyz")); el, x = read_xyz(p); os.remove(p)
        emit(f"Ar_radical_{arname}", el, x, 0, 2)
    if "OA_cation" in CHANNELS and f"X_anion_{s['X']}.com" not in coms:
        p = organic_xyz(s["X_anion"], os.path.join(OUT, f"tmp_{arname}.xyz")); el, x = read_xyz(p); os.remove(p)
        emit(f"X_anion_{s['X']}", el, x, -1, 1)

with open(os.path.join(OUT, "filenames.txt"), "w") as f: f.write("\n".join(coms) + "\n")
with open(os.path.join(OUT, "atom_index.txt"), "w") as f: f.write("\n".join(index_lines) + "\n")

slurm = f"""#!/bin/bash
#SBATCH --partition=notchpeak-guest
#SBATCH --account=owner-guest
#SBATCH --time=4:00:00
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={NPROC}
#SBATCH --mem=18G
#SBATCH -o slurm-%A_%a.out
#SBATCH -e slurm-%A_%a.err
#SBATCH --array=0-{len(coms)-1}%20
# Submit from this folder:  sbatch g16_xtb.slurm filenames.txt

module load gaussian16
module load xtb/6.7.1
export PATH=$HOME/bin:$PATH
which xtb || {{ echo "xtb not found"; exit 1; }}
which xtb-gaussian || {{ echo "xtb-gaussian not found"; exit 1; }}

ulimit -s unlimited
export OMP_STACKSIZE=1G
export OMP_NUM_THREADS={NPROC},1
export OMP_MAX_ACTIVE_LEVELS=1
export MKL_NUM_THREADS={NPROC}

FILELIST=$1
WORKDIR=$PWD
file=$(sed -n "$(( SLURM_ARRAY_TASK_ID + 1 ))p" $FILELIST)
base=${{file%.com}}
export GAUSS_SCRDIR=/scratch/general/vast/$USER/${{SLURM_ARRAY_JOB_ID}}_${{SLURM_ARRAY_TASK_ID}}_${{base}}
mkdir -p $GAUSS_SCRDIR
cp $WORKDIR/$file $GAUSS_SCRDIR/
cd $GAUSS_SCRDIR
g16 < $file > $WORKDIR/${{base}}.log
[[ -f ${{base}}.chk ]] && cp ${{base}}.chk $WORKDIR/
cd $WORKDIR
rm -rf $GAUSS_SCRDIR
"""
with open(os.path.join(OUT, "g16_xtb.slurm"), "w", newline="\n") as f: f.write(slurm)
print(f"Wrote {len(coms)} inputs to {OUT}\n")
print(open(os.path.join(OUT, "atom_index.txt")).read())

Wrote 8 inputs to c:\Users\haruk\Sigman Group Dropbox\Haruka Takenaka\Nickel_CV_project\transition_state\0_generate_xyz\complex_structures\gxtb_opt

terpyNiBr_start_o1_m2	charge +0 mult 2	key {'Ni': 1} trans [Br-N, N-N]
terpyNiBr_OA_PhI_o1_m2	charge +0 mult 2	key {'Ni': 1, 'C_ipso': 2, 'X_leaving': 3} trans [Br-I, C-N, N-N]
terpyNiBr_OA_PhI_o1_m4	charge +0 mult 4	key {'Ni': 1, 'C_ipso': 2, 'X_leaving': 3} trans [Br-I, C-N, N-N]
terpyNiBr_OA_PhI_o3_m2	charge +0 mult 2	key {'Ni': 1, 'C_ipso': 2, 'X_leaving': 3} trans [Br-N, C-I, N-N]  (X trans to Ar: not a direct concerted-OA product)
terpyNiBr_OA_PhI_o3_m4	charge +0 mult 4	key {'Ni': 1, 'C_ipso': 2, 'X_leaving': 3} trans [Br-N, C-I, N-N]  (X trans to Ar: not a direct concerted-OA product)
terpyNiBr_OA_PhI_o5_m2	charge +0 mult 2	key {'Ni': 1, 'C_ipso': 2, 'X_leaving': 3} trans [Br-C, I-N, N-N]
terpyNiBr_OA_PhI_o5_m4	charge +0 mult 4	key {'Ni': 1, 'C_ipso': 2, 'X_leaving': 3} trans [Br-C, I-N, N-N]
ArX_PhI	charge +0 mult 1	



## 7b. Look at the structures
Needs py3Dmol in this environment (`%pip install py3Dmol`, then restart the kernel). Labels show the metal and its donor atoms, numbered as in the Gaussian inputs.

In [32]:
import glob
import py3Dmol

def show_xyz(path, label_atoms="metal_shell", width=450, height=380):
    """label_atoms: "metal_shell" (metal + bound atoms), "heavy", "all", or None."""
    el, x = read_xyz(path)
    v = py3Dmol.view(width=width, height=height)
    v.addModel(open(path).read(), "xyz")
    v.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.22}})
    metal = next((e for e in el if e in GROUP), None)
    if label_atoms == "metal_shell" and metal:
        m, nb, d, _ = metal_environment(el, x, metal); idx = [m] + nb
    elif label_atoms == "heavy":
        idx = [i for i, e in enumerate(el) if e != "H"]
    elif label_atoms:
        idx = range(len(el))
    else:
        idx = []
    for i in idx:
        v.addLabel(f"{el[i]}{i+1}", {"position": {"x": float(x[i][0]), "y": float(x[i][1]), "z": float(x[i][2])},
                                     "fontSize": 11, "fontColor": "black", "backgroundColor": "white",
                                     "backgroundOpacity": 0.7})
    v.zoomTo(); print(os.path.basename(path)); return v.show()

files = sorted(glob.glob(os.path.join(WORKDIR, "gxtb_opt", "*.xyz")))
for i, f in enumerate(files): print(i, os.path.basename(f))

0 ArX_PhI.xyz
1 terpyNiBr_OA_PhI_o1_m2.xyz
2 terpyNiBr_OA_PhI_o1_m4.xyz
3 terpyNiBr_OA_PhI_o3_m2.xyz
4 terpyNiBr_OA_PhI_o3_m4.xyz
5 terpyNiBr_OA_PhI_o5_m2.xyz
6 terpyNiBr_OA_PhI_o5_m4.xyz
7 terpyNiBr_start_o1_m2.xyz


In [38]:
show_xyz(files[6])     # pick a number from the list above

terpyNiBr_OA_PhI_o5_m4.xyz


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 8. Next steps and notes
1. `cd complex_structures/gxtb_opt` and `sbatch g16_xtb.slurm filenames.txt`.
2. For each log: `Normal termination`, `Stationary point found`, and no imaginary frequencies for minima.
3. Compare isomers and spin states **at identical settings** and pick the relevant product for the scan.
4. Concerted OA scan: from the optimized `OA` product, shorten atom 2 (ipso C) – atom 3 (leaving-group atom) toward a
   normal C–X bond (C–I ≈ 2.10, C–Br ≈ 1.90, C–Cl ≈ 1.75, aryl C–O ≈ 1.42 Å): `B 2 3 S <n> -0.050000`.
   Only isomers with X **cis** to Ar can form directly by a three-centered OA.

**Notes**
- `atom_index.txt` lists every input with charge, multiplicity, key atom numbers, and its trans-ligand pattern.
- Multiplicities follow the metal's d-electron count (all ligands treated as closed-shell). The notebook drops any that
  are impossible for the electron count (the "0 3 for an odd-electron system" mistake).
- Weakly bound ligands (OTf, OTs, MeCN, ...) may leave the metal during g-xTB optimization; that is chemistry worth noting.
- `OSET` species reuse the start geometry with one electron removed; `ArX_radanion` reuses the Ar–X geometry. Both are
  starting points for the Marcus-type analysis (optimize, then four-point reorganization energies at DFT).
- molSimplify asks interactively if an output folder exists; the notebook deletes `ms_runs/<name>` before each build.